In [ ]:
#Capacitor Diagonal (Exemplo 6.1)

import numpy as np

# ------------------------------------------------------------
# Exemplo 6.1: Cálculo da capacitância para interface f(x) = h*(x/W)
# com unidades físicas (SI) e impressão em F e pF
# ------------------------------------------------------------

# A constante ε0 é a permissividade do vácuo em (F/m)
ε0 = 8.8541878128e-12  # F/m

# ------------------------------------------------------------
# Função para calcular a capacitância de uma fatia
# A capacitância é dada por C = (ε*A)/h, onde:
# ε é a permissividade do dielétrico,
# A é a área da fatia, e
# h é a espessura da fatia (ou a distância entre as placas para essa fatia)
# ------------------------------------------------------------
def capacitancia(ε, w, h):
    if h == 0:
        h = 1e-10    # Atribui um valor pequeno para evitar a divisão por zero
    c = (ε*w)/h      # Cálculo da capacitância da fatia
    return c         # Retorna a capacitância da fatia

# ------------------------------------------------------------
# Função para associar duas capacitâncias em série
# Quando dois capacitores estão em série, a capacitância equivalente é dada por:
# 1/C_eq = 1/C1 + 1/C2
# ------------------------------------------------------------
def serie(c1, c2):
    c = 1 / (1 / c1 + 1 / c2)  # Cálculo da capacitância equivalente
                               #  de dois capacitores em série
    return c                   # Retorna a capacitância equivalente

# ------------------------------------------------------------
# Função para calcular a posição da interface inclinada f(x)
# f(x) = h*(x/W), onde:
# - h é a separação entre as placas do capacitor
# - x é a posição da fatia
# - W é a largura total das placas
# - L é o Comprimento extrudado
# ------------------------------------------------------------
def f(d, n, i):
    return d * (i / n)  # Calcula a posição da interface
                        #  entre os dois dielétricos para a i-ésima fatia

# ------------------------------------------------------------
# Função para calcular a capacitância analítica, Eq. (12) do manuscrito.
# Equação: C = ε1*ε2*L*W*[*ln(ε1/ε2)]/[h*(ε1 - ε2)]
# ------------------------------------------------------------
def C_analytic_diagonal(ε1, ε2, h, W, L):
    num = ε1*ε2*L*W*np.log(ε1 / ε2)        # Numerador da Eq. analítica
    den = h*(ε1 - ε2)                      # Denominador da Eq. analítica
    return num/den                         # Retorna a capacitância analítica

# ------------------------------------------------------------
# Função principal para calcular a capacitância numérica e analítica
# ------------------------------------------------------------
def main():
    # Parâmetros geométricos em mm (para facilitar a leitura)
    W_mm = 50.0      # Largura da placa em mm
    L_mm = 50.0      # Profundidade em mm
    h_mm = 5.0       # Separação entre as placas em mm

    # Conversão dos parâmetros para metros
    W = W_mm*1e-3  # Converte largura da placa de mm para metros
    L = L_mm*1e-3  # Converte profundidade de mm para metros
    h = h_mm*1e-3  # Converte distância entre as placas de mm para metros

    # Permissividades relativas dos dielétricos
    #  (exemplo: material 1 e material 2)
    k1 = 1.0         # ε1_rel (permissividade relativa do primeiro dielétrico)
    k2 = 5.0         # ε2_rel (permissividade relativa do segundo dielétrico)

    # Conversão para permissividades absolutas [em F/m]
    # (multiplicando pela permissividade do vácuo)
    ε1 = k1*ε0     # Permissividade absoluta do primeiro dielétrico
    ε2 = k2*ε0     # Permissividade absoluta do segundo dielétrico

    # Número de fatias que o capacitor será dividido
    #  (quanto maior, mais preciso será o resultado numérico)
    N = 20000      # Número de fatias para a soma numérica

    # Cálculo da capacitância numérica
    C_num = C_numeric_diagonal(ε1, ε2, h, W, L, N)

    # Cálculo da capacitância analítica
    C_th = C_analytic_diagonal(ε1, ε2, h, W, L)

    # Cálculo do erro relativo entre os valores numérico e analítico
    rel_err = abs(C_num - C_th) / abs(C_th)

    # Exibição dos parâmetros e resultados
    print("Parâmetros:")
    print(f"  W = {W_mm} mm, L = {L_mm} mm, h = {h_mm} mm")
    print(f"  k1 = {k1}, k2 = {k2}")
    print(f"  N = {N} fatias\n")

    # Exibição dos resultados em Farad
    print("Resultados (em Farad):")
    print(f"  C_numérico  = {C_num:.3e} F")
    print(f"  C_analítico = {C_th:.3e} F")

    # Exibição dos Erros
    print(f"  Erro relativo = {rel_err:.3e}\n")

    # Exibição dos resultados em pF (1F = 1e12 pF)
    print("Resultados (em pF):")
    print(f"  C_numérico  = {C_num*1e12:.3e} pF")
    print(f"  C_analítico = {C_th*1e12:.3e} pF")

# ------------------------------------------------------------
# Função para calcular a capacitância numérica via partição em N fatias verticais
# ------------------------------------------------------------
def C_numeric_diagonal(ε1, ε2, h, W, L, N):
    Δx = W/N            # Largura de cada fatia
    C_total = 0.0       # Inicializa a capacitância total

    for k in range(N):  # Loop sobre cada fatia
                        # Ponto médio da fatia
        x = (k + 0.5) * Δx

        # Cálculo de f(x) para cada fatia
        f_val = f(h, N, k)

        # Área da fatia: A = L*Δx
        A = L*Δx

        # Capacitâncias para as camadas em série
        C1 = capacitancia(ε1, A, f_val)      # Capacitância da primeira camada
        C2 = capacitancia(ε2, A, h - f_val)  # Capacitância da segunda camada

        # Capacitância total da fatia (combinação em série)
        C_slice = serie(C1, C2)

        # Somando as fatias (fatias em paralelo)
        C_total += C_slice

    return C_total  # Retorna a capacitância total

# ------------------------------------------------------------
# Executando a função principal
# ------------------------------------------------------------
if __name__ == "__main__":
    main()

Parâmetros:
  W = 50.0 mm, L = 50.0 mm, h = 5.0 mm
  k1 = 1.0, k2 = 5.0
  N = 20000 fatias

Resultados (em Farad):
  C_numérico  = 8.907e-12 F
  C_analítico = 8.906e-12 F
  Erro relativo = 4.971e-05

Resultados (em pF):
  C_numérico  = 8.907e+00 pF
  C_analítico = 8.906e+00 pF
